# Romanized Telugu → Qwen3-4B-Instruct-2507

Colab notebook for reviewing anonymous conversations, mapping speakers, converting to Qwen chat format, and optionally starting QLoRA. Do not train until you verify that assistant messages are yours.

In [ ]:
# Install dependencies
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub


In [ ]:
from google.colab import drive
# Optional: mount Drive if your data is stored there
drive.mount('/content/drive')


## Get the project

Option A: connect Colab to GitHub using **File → Save a copy in GitHub**.

Option B: clone a private repository (replace the URL):

In [ ]:
# Clone public code; private data stays in Drive.
!rm -rf /content/romanized-telugu-qwen-colab
!git clone -q https://github.com/TharunChougoni/romanized-telugu-qwen-colab.git /content/romanized-telugu-qwen-colab
CODE = '/content/romanized-telugu-qwen-colab'


In [ ]:
# Private data lives in Drive. Upload cleaned_conversations.jsonl here first.
PROJECT = '/content/drive/MyDrive/romanized_telugu_dataset_cleaned'
INPUT = f'{PROJECT}/cleaned_conversations.jsonl'
MAPPING = f'{PROJECT}/speaker_mapping.json'
import os
print(INPUT)
print('Drive files:', os.listdir(PROJECT))


## Review anonymous speakers

The cleaner anonymizes speakers separately per conversation/archive. Do not assume speaker_0 is you globally. Inspect a small sample and create a per-conversation mapping if necessary.

In [ ]:
import json, itertools
with open(INPUT, encoding='utf-8') as f:
    samples = list(itertools.islice(f, 5))
for line in samples:
    row=json.loads(line)
    print('CONVERSATION:', row.get('conversation_id'))
    for m in row['messages']:
        print(m['role'], ':', m['content'][:250])
    print('-'*60)


Create `speaker_mapping.json` with the conversation ID and the anonymous speaker that represents you. Example: `{'abc123...': 'speaker_1'}`. Keep this file private.

In [ ]:
# Only run after reviewing samples; this file remains private in Drive.
# Example: mapping = {'conversation_id_here': 'speaker_1'}
mapping = {}
MAPPING = f'{PROJECT}/speaker_mapping.json'
with open(MAPPING, 'w') as f: json.dump(mapping, f, indent=2)
print('Wrote', MAPPING)


## Convert to Qwen chat format

The converter imports `Qwen/Qwen3-4B-Instruct-2507` from Hugging Face and validates its official chat template. It disables thinking mode for natural conversation training.

## Build LoRA-ready prompt/completion examples

This creates one example per assistant turn with prior turns as context. It preserves the edgy/unhinged wording; this stage does not filter profanity, insults, sexual language, sarcasm, or taboo style. It does split train/validation by conversation to reduce leakage.

In [ ]:
!python {CODE}/src/build_lora_dataset.py --input {INPUT} --output {PROJECT}/qwen_lora_sft --all-speakers --max-context-turns 8

In [ ]:
from pathlib import Path
import json
for name in ['train.jsonl','validation.jsonl']:
    p=Path(PROJECT)/'qwen_lora_sft'/name
    print(name, sum(1 for _ in p.open()))
    lines = p.read_text(encoding='utf-8').splitlines()
    print(lines[0][:1000] if lines else '(empty file)')


In [ ]:
# Inspect the first LoRA-ready example.
from pathlib import Path
for name in ['train.jsonl','validation.jsonl']:
    p=Path(PROJECT)/'qwen_lora_sft'/name
    print(name, sum(1 for _ in p.open()))
    lines = p.read_text(encoding='utf-8').splitlines()
    print(lines[0][:1000] if lines else '(empty file)')


## Optional: QLoRA training template

Run only after reviewing the converted JSONL. Start with a small subset and keep the adapter separate.

In [ ]:
# from datasets import load_dataset
# from trl import SFTConfig, SFTTrainer
# from peft import LoraConfig
# MODEL='Qwen/Qwen3-4B-Instruct-2507'
# files={'train':f'{PROJECT}/qwen_lora_sft/train.jsonl','validation':f'{PROJECT}/qwen_lora_sft/validation.jsonl'}
# ds=load_dataset('json', data_files=files)
# cfg=SFTConfig(output_dir=f'{PROJECT}/qwen3-romanized-telugu-lora',max_length=2048,packing=True,assistant_only_loss=True,num_train_epochs=1,per_device_train_batch_size=1,gradient_accumulation_steps=8,gradient_checkpointing=True,eval_strategy='steps',eval_steps=100,logging_steps=10,save_steps=100,report_to='none')
# lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,target_modules='all-linear',task_type='CAUSAL_LM')
# trainer=SFTTrainer(model=MODEL,train_dataset=ds['train'],eval_dataset=ds['validation'],args=cfg,processing_class=__import__('transformers').AutoTokenizer.from_pretrained(MODEL),peft_config=lora)
# trainer.train()
# trainer.save_model(f'{PROJECT}/qwen3-romanized-telugu-lora')


## Privacy and GitHub checklist

- Keep the repository private.
- Never commit raw WhatsApp ZIPs, contact cards, phone numbers, or secrets.
- Treat even cleaned chat text as private.
- Review samples for residual personal details before training.
- Do not upload the final adapter publicly without consent.